# Image Corruption Detector — EDA & Results Analysis

This notebook walks through the dataset, visualizes each corruption type, and analyzes training results.

**Contents:**
1. Setup & imports
2. Dataset statistics
3. Corruption type examples
4. Training results
5. Confusion matrix analysis
6. Sample predictions
7. Model performance summary

## 1. Setup & Imports

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image

# Make sure the project root is on the path
ROOT = Path('.').resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.corruption_pipeline import apply_corruption, list_corruption_types
from src.data.dataset import CLASS_NAMES, CLASS_TO_IDX
from src.utils.visualization import (
    plot_training_curves,
    plot_confusion_matrix,
    plot_sample_predictions,
    plot_corruption_examples,
    plot_class_distribution,
)

sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams['figure.dpi'] = 120
print('Setup complete.')

## 2. Dataset Statistics

We load the metadata CSV and inspect the class distribution across splits.

In [ ]:
METADATA_CSV = ROOT / 'data' / 'metadata.csv'

if not METADATA_CSV.exists():
    print('Dataset not generated yet. Run: python src/data/generate_dataset.py')
else:
    df = pd.read_csv(METADATA_CSV)
    print(f'Total samples: {len(df):,}')
    print(f'Splits: {df["split"].value_counts().to_dict()}')
    print(f'Classes: {df["corruption_type"].nunique()}')
    display(df.head(5))

In [ ]:
if METADATA_CSV.exists():
    # Class distribution across all splits
    counts = df['corruption_type'].value_counts().reindex(CLASS_NAMES).fillna(0).astype(int)
    fig, ax = plt.subplots(figsize=(11, 4))
    bars = ax.bar(counts.index, counts.values, color=sns.color_palette('tab10', len(counts)))
    ax.set_title('Sample Distribution by Corruption Type (all splits)', fontsize=13, fontweight='bold')
    ax.set_xlabel('Corruption Type')
    ax.set_ylabel('Number of Samples')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                str(int(bar.get_height())), ha='center', fontsize=9)
    plt.xticks(rotation=20, ha='right')
    plt.tight_layout()
    plt.show()

In [ ]:
if METADATA_CSV.exists():
    # Per-split breakdown
    split_class = df.groupby(['split', 'corruption_type']).size().unstack(fill_value=0)
    split_class = split_class.reindex(columns=CLASS_NAMES)
    display(split_class)

## 3. Corruption Type Examples

Visual comparison of each corruption at all three severity levels.

In [ ]:
# Generate a sample image to corrupt (or use one from the dataset)
rng = np.random.default_rng(42)
sample = rng.integers(40, 200, size=(64, 64, 3), dtype=np.uint8)
# Attempt to load a real image if available
if METADATA_CSV.exists():
    clean_row = df[df['corruption_type'] == 'clean'].iloc[0]
    sample = np.array(Image.open(clean_row['filepath']).convert('RGB'))

corruption_types = [c for c in list_corruption_types() if c != 'clean']
severities = [1, 2, 3]

fig, axes = plt.subplots(len(corruption_types), len(severities) + 1,
                          figsize=(3*(len(severities)+1), 2.5*len(corruption_types)))
fig.suptitle('Corruption Types × Severity Levels', fontsize=14, fontweight='bold')

for row_i, ctype in enumerate(corruption_types):
    axes[row_i, 0].imshow(sample)
    axes[row_i, 0].set_title('Clean' if row_i == 0 else '', fontsize=8)
    axes[row_i, 0].set_ylabel(ctype, fontsize=9, fontweight='bold')
    axes[row_i, 0].axis('off')
    for col_i, sev in enumerate(severities):
        corrupted = apply_corruption(sample, ctype, sev)
        axes[row_i, col_i + 1].imshow(corrupted)
        if row_i == 0:
            axes[row_i, col_i + 1].set_title(f'Severity {sev}', fontsize=8)
        axes[row_i, col_i + 1].axis('off')

plt.tight_layout()
plt.show()

## 4. Training Results

Load the training history JSON and plot learning curves.

In [ ]:
HISTORY_PATH = ROOT / 'outputs' / 'training_history.json'

if not HISTORY_PATH.exists():
    print('Training history not found. Run scripts/train.py first.')
else:
    with open(HISTORY_PATH) as f:
        history = json.load(f)

    epochs = range(1, len(history['train_loss']) + 1)
    best_epoch = int(np.argmax(history['val_acc'])) + 1
    best_val_acc = max(history['val_acc'])
    print(f'Best validation accuracy: {best_val_acc:.4f} at epoch {best_epoch}')
    print(f'Total epochs trained: {len(epochs)}')

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle('Training History', fontsize=14, fontweight='bold')

    axes[0].plot(epochs, history['train_loss'], label='Train', lw=2)
    axes[0].plot(epochs, history['val_loss'], label='Validation', lw=2, ls='--')
    axes[0].axvline(best_epoch, color='gray', ls=':', label=f'Best epoch ({best_epoch})')
    axes[0].set_title('Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Cross-Entropy Loss')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(epochs, history['train_acc'], label='Train', lw=2)
    axes[1].plot(epochs, history['val_acc'], label='Validation', lw=2, ls='--')
    axes[1].axvline(best_epoch, color='gray', ls=':', label=f'Best epoch ({best_epoch})')
    axes[1].set_title('Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    axes[1].set_ylim([0, 1])

    plt.tight_layout()
    plt.show()

## 5. Confusion Matrix Analysis

In [ ]:
METRICS_PATH = ROOT / 'outputs' / 'eval' / 'test_metrics.json'

if not METRICS_PATH.exists():
    print('Evaluation metrics not found. Run scripts/evaluate.py first.')
else:
    with open(METRICS_PATH) as f:
        metrics = json.load(f)

    cm = np.array(metrics['confusion_matrix'])
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for ax, data, fmt, title in zip(
        axes,
        [cm_norm, cm],
        ['.2f', 'd'],
        ['Normalized (Recall)', 'Raw Counts'],
    ):
        sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                    linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
        ax.set_title(f'Confusion Matrix — {title}', fontsize=12, fontweight='bold')
        ax.set_xlabel('Predicted', fontsize=11)
        ax.set_ylabel('True', fontsize=11)
        plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
        plt.setp(ax.get_yticklabels(), rotation=0)

    plt.suptitle('Test Set Confusion Matrices', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print(f"\nTest Accuracy : {metrics['accuracy']:.4f}")
    print(f"Test Macro F1 : {metrics['macro_f1']:.4f}")

## 6. Per-Class Performance

In [ ]:
if METRICS_PATH.exists():
    per_class = metrics['per_class']
    rows = []
    for cls, vals in per_class.items():
        rows.append({'Class': cls, **{k.capitalize(): v for k, v in vals.items()}})
    df_pc = pd.DataFrame(rows).set_index('Class')
    display(df_pc.style.format({'Precision': '{:.4f}', 'Recall': '{:.4f}',
                                 'F1': '{:.4f}', 'Support': '{:d}'})
            .background_gradient(subset=['Precision', 'Recall', 'F1'], cmap='YlGn'))

## 7. Model Performance Summary

Summary table of key results.

In [ ]:
summary = {
    'Architecture': 'ResNet-18 (pretrained ImageNet)',
    'Task': '7-class corruption classification',
    'Input Size': '224 × 224 × 3',
    'Frozen Layers': 'conv1, bn1, layer1, layer2',
    'Fine-tuned Layers': 'layer3, layer4, classifier head',
    'Classifier Head': '512 → 256 (ReLU, Dropout 0.3) → 7',
}

if METRICS_PATH.exists():
    summary['Test Accuracy'] = f"{metrics['accuracy']:.4f}"
    summary['Test Macro F1'] = f"{metrics['macro_f1']:.4f}"

if HISTORY_PATH.exists():
    summary['Best Val Accuracy'] = f"{best_val_acc:.4f} (epoch {best_epoch})"
    summary['Epochs Trained'] = len(history['train_acc'])

for k, v in summary.items():
    print(f"{k:<25}: {v}")